# CardioIA — classificação textual de risco com TF-IDF

Notebook da entrega obrigatória da Fase 2. A base é simulada e o resultado é exclusivamente acadêmico: não representa diagnóstico ou triagem clínica real.

## 1. Objetivo e rastreabilidade

O experimento transforma frases curtas em vetores TF-IDF e treina uma Regressão Logística para distinguir os rótulos simulados **baixo risco** e **alto risco**.

As versões feminina e masculina de um mesmo cenário compartilham o mesmo `id_cenario`. A divisão é feita por esse identificador para impedir que versões quase equivalentes apareçam simultaneamente em treino e teste.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

RAIZ = Path.cwd()
if not (RAIZ / "fase2").exists():
    RAIZ = Path.cwd().parents[1]

sys.path.insert(0, str(RAIZ / "fase2" / "src"))

from classificar_risco_texto import (
    analisar_vies_contrafactual,
    avaliar_modelo,
    carregar_dataset,
    criar_pipeline,
    dividir_por_cenario,
    termos_mais_influentes,
)

## 2. Dataset simulado

O arquivo possui frases rotuladas e pares contrafactuais por sexo. Os rótulos foram definidos apenas para demonstrar o pipeline solicitado no enunciado.

In [2]:
dados = carregar_dataset()
resumo_dataset = pd.DataFrame({
    "total_frases": [len(dados)],
    "cenarios_unicos": [dados["id_cenario"].nunique()],
    "duplicadas": [int(dados["frase"].duplicated().sum())],
})
display(resumo_dataset)
display(pd.crosstab(dados["situacao"], dados["sexo_referencia"]))

,total_frases,cenarios_unicos,duplicadas
0,120,60,0


sexo_referencia,feminino,masculino
situacao,,
alto risco,30,30
baixo risco,30,30


## 3. Separação sem vazamento por cenário

A divisão reserva 20% dos cenários de cada classe para teste. As duas versões demográficas permanecem sempre no mesmo conjunto.

In [3]:
treino, teste = dividir_por_cenario(dados)
assert not set(treino["id_cenario"]).intersection(teste["id_cenario"])

display(pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "frases": [len(treino), len(teste)],
    "cenarios": [treino["id_cenario"].nunique(), teste["id_cenario"].nunique()],
}))
display(pd.crosstab(teste["situacao"], teste["sexo_referencia"]))

,conjunto,frases,cenarios
0,treino,96,48
1,teste,24,12


sexo_referencia,feminino,masculino
situacao,,
alto risco,6,6
baixo risco,6,6


## 4. TF-IDF e Regressão Logística

O TF-IDF e o classificador ficam no mesmo `Pipeline`. Assim, o vocabulário e os pesos são ajustados apenas com as frases de treino.

In [4]:
modelo = criar_pipeline()
modelo.fit(treino["frase"], treino["situacao"])

vetorizador = modelo.named_steps["tfidf"]
pd.DataFrame({
    "configuracao": ["Vocabulário", "N-gramas", "Classes"],
    "valor": [
        len(vetorizador.vocabulary_),
        str(vetorizador.ngram_range),
        ", ".join(modelo.classes_),
    ],
})

Out[0]: 
  configuracao                    valor
0  Vocabulário                      609
1     N-gramas                   (1, 2)
2      Classes  alto risco, baixo risco


## 5. Avaliação no conjunto de teste

Além da acurácia solicitada, são calculados precisão, recall, F1 e ROC AUC. Em triagem, observar apenas a acurácia pode esconder erros importantes na classe de maior risco.

In [5]:
metricas, previsoes = avaliar_modelo(modelo, teste)
metricas_resumo = {
    chave: valor
    for chave, valor in metricas.items()
    if chave != "relatorio_classificacao"
}
display(pd.DataFrame([metricas_resumo]).style.format({
    "acuracia": "{:.3f}",
    "precisao_alto_risco": "{:.3f}",
    "recall_alto_risco": "{:.3f}",
    "f1_alto_risco": "{:.3f}",
    "roc_auc": "{:.3f}",
}))
display(pd.DataFrame(metricas["relatorio_classificacao"]).T)

,n_treino,n_teste,acuracia,precisao_alto_risco,recall_alto_risco,f1_alto_risco,roc_auc
0,None,24,1.000,1.000,1.000,1.000,1.000


,precision,recall,f1-score,support
alto risco,1.0,1.0,1.0,12.0
baixo risco,1.0,1.0,1.0,12.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,24.0
weighted avg,1.0,1.0,1.0,24.0


In [6]:
matriz = confusion_matrix(
    teste["situacao"],
    previsoes["predicao"],
    labels=["baixo risco", "alto risco"],
)
ConfusionMatrixDisplay(
    matriz,
    display_labels=["Baixo risco", "Alto risco"],
).plot(cmap="RdPu", colorbar=False)
plt.title("Matriz de confusão — conjunto de teste")
plt.show()

## 6. Termos associados às classes

Os coeficientes descrevem associações aprendidas nesta base simulada. Eles não demonstram causalidade ou relevância médica universal.

In [7]:
termos = termos_mais_influentes(modelo, quantidade=12)
display(termos.groupby("direcao", group_keys=False).head(12))

,termo,coeficiente,direcao
0,leve,0.774063,baixo risco
1,melhora,0.636377,baixo risco
2,depois,0.627372,baixo risco
3,apos,0.619350,baixo risco
4,depois de,0.556128,baixo risco
5,sem,0.539766,baixo risco
6,que melhora,0.482497,baixo risco
7,horas,0.392007,baixo risco
8,relata leve,0.387441,baixo risco
9,desconforto,0.370315,baixo risco


## 7. Análise de viés contrafactual

Comparamos versões feminina e masculina do mesmo relato. Como sintomas e rótulo são mantidos, uma grande variação indicaria sensibilidade indevida ao marcador textual de sexo.

Essa análise complementa — e não substitui — a avaliação tabular por sexo já existente no projeto.

In [8]:
resumo_vies, pares = analisar_vies_contrafactual(modelo, teste)
display(pd.DataFrame([resumo_vies]).style.format("{:.4f}"))
display(pares.sort_values("diferenca_absoluta", ascending=False).head(10))

,media_probabilidade_feminino,media_probabilidade_masculino,diferenca_media_absoluta,maior_diferenca_absoluta
0,0.5062,0.5088,0.0025,0.0032


sexo_referencia,id_cenario,situacao,feminino,masculino,diferenca_absoluta
11,baixo_24,baixo risco,0.421482,0.424732,0.003250
5,alto_27,alto risco,0.613303,0.616400,0.003097
6,baixo_12,baixo risco,0.333766,0.336848,0.003082
3,alto_20,alto risco,0.593823,0.596637,0.002814
1,alto_11,alto risco,0.556965,0.559530,0.002565
7,baixo_15,baixo risco,0.358784,0.361314,0.002530
2,alto_15,alto risco,0.521502,0.524022,0.002520
10,baixo_21,baixo risco,0.425549,0.427948,0.002399
9,baixo_17,baixo risco,0.478042,0.480396,0.002355
4,alto_23,alto risco,0.608571,0.610883,0.002312


## 8. Conclusões e limitações

- O pipeline exigido de TF-IDF e classificação textual foi implementado e avaliado em dados não vistos.
- A separação por cenário impede vazamento entre pares contrafactuais.
- O teste por sexo verifica estabilidade textual em exemplos equivalentes.
- A base é pequena, balanceada e totalmente simulada; seu balanceamento não representa prevalência.
- Os dados textuais, tabulares e visuais têm origens distintas e permanecem como modalidades independentes.
- O modelo não possui validação clínica, externa ou prospectiva e não deve ser usado em atendimentos reais.